In [0]:
df_order = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_orders_dataset.csv")

In [0]:
df_geoloacation = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_geolocation_dataset.csv")

In [0]:
df_customer = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_customers_dataset.csv")

In [0]:
df_order_item = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_order_items_dataset.csv")

In [0]:
df_product = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_products_dataset.csv")

In [0]:
df_seller = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_sellers_dataset.csv")

In [0]:
df_full_order = df_order.join(df_order_item,'order_id','inner')

In [0]:
df_full_order = df_full_order.join(df_product,'product_id','inner')


In [0]:
df_full_order = df_full_order.join(df_seller,'seller_id','inner')


In [0]:
df_full_order = df_full_order.join(df_customer,'customer_id','inner')


In [0]:
df_review = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_order_reviews_dataset.csv")

In [0]:
df_payment = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/my_first_pyspark_project/bronze_layer/csv_tables/olist_order_payments_dataset.csv")

In [0]:
df_full_order.printSchema()

In [0]:
df_full_order = df_full_order.join(df_geoloacation,df_full_order.customer_zip_code_prefix == df_geoloacation.geolocation_zip_code_prefix,'left')

In [0]:
df_full_order = df_full_order.join(df_review,'order_id','left')


In [0]:
df_full_order = df_full_order.join(df_payment,'order_id','left')


In [0]:
df_full_order.printSchema()

In [0]:
from pyspark.sql.functions import *
# Calculate total price for each seller
df_full_order.groupBy('seller_id').agg(round(sum('price'),2).alias('total_price')).orderBy(desc('total_price')).display()

In [0]:
# Top 10 most sold product
df_full_order.groupBy('product_id').agg(count('product_id').alias('total_sold')).orderBy(desc('total_sold')).limit(10).display()


In [0]:
# Total Revenue & Average order value (AOV) per customer
customer_metrix = df_full_order.groupBy('customer_id').agg(round(sum('price'),2).alias('total_revenue'),round(avg('price'),2).alias('AOV')).orderBy(desc('total_revenue'))
customer_metrix.display()


In [0]:
# Seller Performance Metrics
seller_performance = df_full_order.groupBy('seller_id').agg(count('order_id').alias('total_order'),round(sum('price'),2).alias('total_revenue'),round(avg('review_score')).alias('Average_review_score'),round(stddev('price'),2).alias('price_variability')).orderBy(desc('total_revenue'))
seller_performance.display()

In [0]:
# Product Popularity Metrics
product_metric = df_full_order.groupBy('product_id').agg(count('order_id').alias('total_order'),sum('price').alias('total_revenue'),round(avg('price'),2).alias('Average_Price'),collect_set('seller_id').alias('unique_seller')).orderBy(desc('total_order'))
product_metric.display()

In [0]:
# Monthly Matrix as per Year
Monthly_order = df_full_order.groupBy(year('order_purchase_timestamp').alias('Year'),month('order_purchase_timestamp').alias('Month')).agg(count('order_id').alias('total_order'),round(sum('price'),2).alias('total_revenue'),round(avg('price'),2).alias('Average_order_value')).orderBy('year','month')
Monthly_order.display()

In [0]:
# Order_revenue
df_full_order = df_full_order.withColumn('Order_Revenue',col('price') + col('freight_value'))
df_full_order.display()

In [0]:
df_full_order.select('price','freight_value','Order_Revenue').display()

In [0]:
# Customer Segmentation
customer_segment = customer_metrix.withColumn('customer_segment',when(col('AOV') >= 1300, 'High-value').when((col('AOV') < 1300) & (col('AOV') > 500), 'Medium-value').otherwise('Low-value'))
customer_segment.display()

In [0]:
df_full_order = df_full_order.join(customer_segment.select('customer_id','customer_segment'),'customer_id','left')
df_full_order.display()


In [0]:
# Weekday Orders Vs Weeday Orders
df_full_order = df_full_order.withColumn('order_day_type',when(dayofweek(col('order_purchase_timestamp')).isin(1,7),'Weekend').otherwise('Weekday'))
df_full_order.display()

In [0]:
df_full_order.write.format('delta').mode('overwrite').save('/Volumes/my_first_pyspark_project/delta_table/Full_order_details')


In [0]:
# Optimize the table
spark.sql("""
OPTIMIZE delta.`/Volumes/my_first_pyspark_project/delta_table/detla_table`
ZORDER BY (customer_id)
""")
